# Figure 1 — Perturb-CITE-seq Screen Design and Validation
Nature-quality figures. Panels C–G.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from matplotlib import rcParams
from scipy.stats import spearmanr
from adjustText import adjust_text

rcParams['pdf.fonttype'] = 42
rcParams['ps.fonttype'] = 42
rcParams['pdf.use14corefonts'] = True
warnings.filterwarnings(action='ignore')
sc.set_figure_params(figsize=[4, 4], fontsize=12, dpi=100, frameon=False)


In [ ]:
COND_COLORS = {'Low': '#e1812c', 'Control': '#3274a1', 'High': '#3a923a'}
COND_ORDER = ['Low', 'Control', 'High']

OUT_DIR = '../figures/nature_figures/Fig1'
os.makedirs(OUT_DIR, exist_ok=True)

def save_fig(name, formats=('pdf', 'png')):
    for fmt in formats:
        plt.savefig(os.path.join(OUT_DIR, f'{name}.{fmt}'), bbox_inches='tight', dpi=300)
    print(f'Saved: {name}')


In [ ]:
BASE = '/home/wangh256/hanchen/Pert_PG/perturb-me/PerturbME_transfer/PerturbCITE_ICR/202008_full_exp'
MAGECK_DIR = os.path.join(BASE, 'guide_seq/mageck_out')

adata = sc.read_h5ad(os.path.join(BASE, 'adata_RNA_CITE.h5ad'))
adata.obs['Condition'] = pd.Categorical(adata.obs['Condition'].astype(str), categories=COND_ORDER)
condition_arr = adata.obs['Condition'].to_numpy().astype(str)

print(f'Loaded: {adata.shape[0]:,} cells × {adata.shape[1]:,} features')
print({c: int((condition_arr == c).sum()) for c in COND_ORDER})


## Fig 1B — HLA Protein Distribution with FACS Tails
Histogram of CITE-seq HLA-A,B,C protein across all cells; bottom 5% (Low) and top 5% (High) tails highlighted.

In [ ]:
from matplotlib.ticker import MultipleLocator

hla = adata[:, 'CITE-HLA_A'].X
hla = hla.toarray().flatten() if hasattr(hla, 'toarray') else hla.flatten()

p_low  = np.percentile(hla, 5)
p_high = np.percentile(hla, 95)

MIDDLE_COLOR = '#3a3a3a'
bins = np.linspace(hla.min(), hla.max(), 68)
counts, edges = np.histogram(hla, bins=bins)
centers = (edges[:-1] + edges[1:]) / 2
width = (edges[1] - edges[0]) * 1.2

bar_colors = np.where(
    centers <= p_low,  COND_COLORS['Low'],
    np.where(centers >= p_high, COND_COLORS['High'], MIDDLE_COLOR),
)

fig, ax = plt.subplots(figsize=(3.5, 3))
ax.bar(centers, counts, width=width, color=bar_colors, edgecolor='white', linewidth=0.15)

ax.axvline(p_low,  linestyle='--', color='gray', linewidth=1)
ax.axvline(p_high, linestyle='--', color='gray', linewidth=1)

ymax = counts.max()
# ax.text(p_low,  ymax * 0.97, f'  5th pct\n  {p_low:.2f}',  ha='left',  va='top', fontsize=10, color='gray')
# ax.text(p_high, ymax * 0.97, f'95th pct  \n{p_high:.2f}  ', ha='right', va='top', fontsize=10, color='gray')

# ── Sawtooth-style dense ticks, no tick labels ─────────────────
x_range = hla.max() - hla.min()
ax.xaxis.set_major_locator(MultipleLocator(x_range / 10))
ax.xaxis.set_minor_locator(MultipleLocator(x_range / 50))
ax.yaxis.set_major_locator(MultipleLocator(ymax / 5))
ax.yaxis.set_minor_locator(MultipleLocator(ymax / 25))

ax.tick_params(which='major', length=5, width=0.9, direction='out', color='black')
ax.tick_params(which='minor', length=3, width=0.5, direction='out', color='dimgray')
ax.set_xticklabels([])
ax.set_yticklabels([])

ax.set_xlabel('HLA-A,B,C protein expression (CITE-seq)')
ax.set_ylabel('Number of cells')
# ax.set_title(f'HLA protein distribution across all cells (n={len(hla):,})')

sns.despine()
ax.grid(False)
plt.tight_layout()
save_fig('Fig1B_HLA_histogram_with_tails')
plt.show()


## Fig 1C — UMAP by Condition

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, cond in zip(axes, COND_ORDER):
    sc.pl.umap(
        adata, color='Condition', groups=[cond], palette=COND_COLORS,
        ax=ax, show=False, size=1, frameon=False, legend_loc=None,
        title=f'{cond} (n={(condition_arr == cond).sum():,})',
    )
plt.tight_layout()
save_fig('Fig1C_UMAP_by_condition')
plt.show()


## Fig 1D — HLA-A,B,C Protein Expression (UMAP)

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
sc.pl.umap(
    adata, color='CITE-HLA_A', vmin=2, vmax=6, size=1, cmap='magma',
    ax=ax, show=False, frameon=False, title='HLA-A,B,C protein',
)
plt.tight_layout()
save_fig('Fig1D_HLA_protein_UMAP')
plt.show()


## Fig 1E — HLA-A Expression: Protein vs RNA

In [ ]:
def extract_expr(feature):
    x = adata[:, feature].X
    return x.toarray().flatten() if hasattr(x, 'toarray') else x.flatten()

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, feature, title in [
    (axes[0], 'CITE-HLA_A', 'Protein (CITE-seq)'),
    (axes[1], 'HLA-A',      'mRNA (scRNA-seq)'),
]:
    df = pd.DataFrame({'Expression': extract_expr(feature), 'Condition': condition_arr})
    sns.violinplot(data=df, y='Expression', x='Condition', ax=ax,
                   order=COND_ORDER, palette=COND_COLORS, cut=0, inner='box',
                   saturation=0.8)
    for coll in ax.collections:
        coll.set_alpha(0.7)
    ax.set_title(title)
    ax.set_xlabel('')
sns.despine()
for _ax in np.asarray(axes).flat: _ax.grid(False)
plt.tight_layout()
save_fig('Fig1E_RNA_vs_protein_violin')
plt.show()


## Fig 1F — sgRNA Log2 Fold Change Rank View

In [ ]:
low_input_df  = pd.read_csv(os.path.join(MAGECK_DIR, 'Low_vs_Input.sgrna_summary.txt'),  sep='\t')
high_input_df = pd.read_csv(os.path.join(MAGECK_DIR, 'High_vs_Input.sgrna_summary.txt'), sep='\t')


def sgRankView(df, ax, title, top_n=15, bottom_n=5,
               pos_color='#c0392b', neg_color='#3274a1'):
    """MAGeCKFlute-style sgRankView: red=enriched (+LFC), blue=depleted (-LFC)."""
    df = df[~df['Gene'].str.contains('NO_SITE|NON-GENE', na=False)].copy()
    gene_median = df.groupby('Gene')['LFC'].median().sort_values()
    gene_order = pd.concat([gene_median.head(bottom_n), gene_median.tail(top_n)]).index.tolist()

    all_lfc = df[df['Gene'].isin(gene_order)]['LFC'].values
    x_pad = (all_lfc.max() - all_lfc.min()) * 0.08
    x_min, x_max = all_lfc.min() - x_pad, all_lfc.max() + x_pad

    for i, gene in enumerate(gene_order):
        ax.barh(i, x_max - x_min, left=x_min, color='#ebebeb', edgecolor='#cccccc', height=0.65)
        for lfc in df.loc[df['Gene'] == gene, 'LFC'].values:
            ax.plot([lfc, lfc], [i - 0.25, i + 0.25],
                    color=pos_color if lfc >= 0 else neg_color,
                    linewidth=2, solid_capstyle='round')

    ax.set_yticks(range(len(gene_order)))
    ax.set_yticklabels(gene_order)
    ax.set_xlabel('Log$_2$(fold change)')
    ax.set_title(title)
    ax.set_xlim([x_min, x_max])
    ax.tick_params(axis='y', length=0)


fig, axes = plt.subplots(1, 2, figsize=(10, 6))
sgRankView(low_input_df,  axes[0], 'HLA-Low vs Input')
sgRankView(high_input_df, axes[1], 'HLA-High vs Input')
sns.despine()
for _ax in np.asarray(axes).flat: _ax.grid(False)
plt.tight_layout()
save_fig('Fig1F_sgRNA_LFC_rankview')
plt.show()


## Fig 1G — Bulk Reads vs Single-Cell Cells per Target

In [ ]:
reads_per_gene_high = np.load(os.path.join(MAGECK_DIR, 'reads_per_gene_high.npy'))
reads_per_gene_low  = np.load(os.path.join(MAGECK_DIR, 'reads_per_gene_low.npy'))
cells_per_gene_high = np.load(os.path.join(MAGECK_DIR, 'cells_per_gene_high.npy'))
cells_per_gene_low  = np.load(os.path.join(MAGECK_DIR, 'cells_per_gene_low.npy'))
all_genes           = np.load(os.path.join(MAGECK_DIR, 'all_genes.npy'))

control_mask = np.isin(all_genes, ['NO_SITE', 'ONE_NON-GENE_SITE'])
gene_mask    = ~control_mask

configs = [
    ('Low',  reads_per_gene_low,  cells_per_gene_low,  (0, 3e5),  (0, 200), 40),
    ('High', reads_per_gene_high, cells_per_gene_high, (0, 150e3), (0, 100), 25),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (cond, reads, cells, xlim, ylim, thr) in zip(axes, configs):
    r_g, c_g = reads[gene_mask], cells[gene_mask]
    ax.scatter(r_g, c_g, s=12, c='#4a4a4a', alpha=0.5, rasterized=True, linewidths=0)

    rho, _ = spearmanr(r_g, c_g)
    ax.text(0.05, 0.95, f'$\\rho$ = {rho:.3f}', transform=ax.transAxes, va='top')
    ax.set_xlabel('Reads per target')
    ax.set_ylabel('Cells per target')
    ax.set_title(f'HLA-{cond}')
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    label_inds = np.where((cells > thr) & gene_mask)[0]
    label_inds = label_inds[~np.isin(label_inds, np.where(control_mask)[0])]
    texts = [ax.text(reads[i], cells[i], all_genes[i], fontsize=12, fontstyle='italic') for i in label_inds]
    if texts:
        adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='#c0392b', lw=0.8))
sns.despine()
for _ax in np.asarray(axes).flat: _ax.grid(False)
plt.tight_layout()
save_fig('Fig1G_reads_vs_cells_per_target')
plt.show()
